### Vitterhetsakademin.ipynb
* hämtar alla objekt med P3389
* bygger gamla URL
* testar id.html
* följer redirect
* extraherar slug
* sparar resultat
* genererar QuickStatements

In [1]:
from datetime import datetime
start_time  = datetime.now()
print("Last run: ", start_time)

Last run:  2026-03-05 05:10:05.318945


In [2]:
import requests
import pandas as pd
import urllib.parse
import time

from bs4 import BeautifulSoup
from tqdm import tqdm
from rapidfuzz import fuzz

# -----------------------------
# Session
# -----------------------------

session = requests.Session()
session.headers.update({
    "User-Agent": "Wikidata Vitterhets scraper (salgo60@msn.com)"
})

# -----------------------------
# SPARQL
# -----------------------------

SPARQL_URL = "https://query.wikidata.org/sparql"

QUERY = """
SELECT ?person ?personLabel ?birth ?id WHERE {
  ?person wdt:P3389 ?id .
  OPTIONAL { ?person wdt:P569 ?birth }
  SERVICE wikibase:label { bd:serviceParam wikibase:language "sv,en". }
}
"""

print("Fetching Wikidata...")

r = session.get(
    SPARQL_URL,
    params={"query": QUERY, "format": "json"}
)

data = r.json()

rows = []

for b in data["results"]["bindings"]:

    qid = b["person"]["value"].split("/")[-1]
    label = b["personLabel"]["value"]

    birth = None
    if "birth" in b:
        birth = b["birth"]["value"][:4]

    rows.append({
        "qid": qid,
        "label": label,
        "birth": birth,
        "wd_id": b["id"]["value"]
    })

df_wd = pd.DataFrame(rows)

print("WD items:", len(df_wd))

# -----------------------------
# Search crawler
# -----------------------------

def search_all(query):

    urls = set()
    start = 0

    while True:

        q = urllib.parse.quote(query)

        url = f"https://www.vitterhetsakademien.se/ovrigt/sokresultat.html?query={q}&startAtHit={start}"

        r = session.get(url)

        soup = BeautifulSoup(r.text, "html.parser")

        found = 0

        for a in soup.find_all("a", href=True):

            href = a["href"]

            if "/ledamoter/ledamoter/" in href:

                urls.add(
                    urllib.parse.urljoin(url, href)
                )

                found += 1

        if found == 0:
            break

        start += 20

    return urls


alphabet = list("abcdefghijklmnopqrstuvwxyzåäö")

slug_urls = set()

print("Crawling site search...")

for letter in tqdm(alphabet):

    urls = search_all(letter)

    slug_urls.update(urls)

print("Slug pages found:", len(slug_urls))

# -----------------------------
# Scrape person pages
# -----------------------------

people = []

print("Scraping person pages...")

for url in tqdm(slug_urls):

    try:

        r = session.get(url)

        soup = BeautifulSoup(r.text, "html.parser")

        title = soup.title.text.split("|")[0].strip()

        text = soup.get_text()

        birth = None

        if "f." in text:
            try:
                birth = text.split("f.")[1][:10].strip()
            except:
                pass

        people.append({
            "url": url,
            "name": title,
            "birth": birth
        })

    except:
        pass

df_people = pd.DataFrame(people)

print("Persons scraped:", len(df_people))

# -----------------------------
# Match against Wikidata
# -----------------------------

matches = []

print("Matching persons...")

for _, wd in tqdm(df_wd.iterrows(), total=len(df_wd)):

    best_score = 0
    best_row = None

    for _, p in df_people.iterrows():

        score = fuzz.token_set_ratio(wd["label"], p["name"])

        if score > best_score:
            best_score = score
            best_row = p

    if best_score > 85:

        matches.append({
            "qid": wd["qid"],
            "url": best_row["url"],
            "score": best_score
        })

print("Matches found:", len(matches))

# -----------------------------
# QuickStatements
# -----------------------------

qs = []

for m in matches:

    qs.append(
        f'{m["qid"]}\tP973\t"{m["url"]}"'
    )

with open("quickstatements_vitterhets.txt","w",encoding="utf-8") as f:

    f.write("\n".join(qs))

# -----------------------------
# Save debug data
# -----------------------------

df_people.to_csv("vitterhets_people.csv", index=False)

df_wd.to_csv("wikidata_input.csv", index=False)

print("Files saved:")
print("quickstatements_vitterhets.txt")
print("vitterhets_people.csv")
print("wikidata_input.csv")

Fetching Wikidata...
WD items: 1410
Crawling site search...


100%|███████████████████████████████████████████| 29/29 [00:26<00:00,  1.10it/s]


Slug pages found: 590
Scraping person pages...


100%|█████████████████████████████████████████| 590/590 [01:37<00:00,  6.03it/s]


Persons scraped: 590
Matching persons...


100%|██████████████████████████████████████| 1410/1410 [00:09<00:00, 142.86it/s]

Matches found: 320
Files saved:
quickstatements_vitterhets.txt
vitterhets_people.csv
wikidata_input.csv


In [3]:
df_people = pd.DataFrame(people)

print("Persons scraped:", len(df_people))

Persons scraped: 590


In [4]:
df_people.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 590 entries, 0 to 589
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   url     590 non-null    object
 1   name    590 non-null    object
 2   birth   590 non-null    object
dtypes: object(3)
memory usage: 14.0+ KB


In [6]:
print("Saved files:")
print("vitterhets_wd_check.csv")
print("quickstatements.txt")
print("vitterhets_wd_check.json")

Saved files:
vitterhets_wd_check.csv
quickstatements.txt
vitterhets_wd_check.json


In [7]:
found = 0
not_found = 0

results = []

for _, row in tqdm(df_wd.iterrows(), total=len(df_wd), desc="Testing search"):

    name = row["label"]

    query = urllib.parse.quote(name)

    url = f"https://www.vitterhetsakademien.se/ovrigt/sokresultat.html?query={query}&submitButton=S%C3%B6k"

    r = session.get(url)

    soup = BeautifulSoup(r.text, "html.parser")

    slug = None

    for a in soup.find_all("a", href=True):

        href = a["href"]

        if "/ledamoter/ledamoter/" in href:

            slug = urllib.parse.urljoin(url, href)
            break

    if slug:
        found += 1
    else:
        not_found += 1

    results.append({
        "qid": row["qid"],
        "name": name,
        "url": slug
    })

print("Found:", found)
print("Not found:", not_found)
print("Coverage:", round(found/len(df_wd)*100,2), "%")

Testing search: 100%|███████████████████████| 1410/1410 [02:46<00:00,  8.48it/s]

Found: 1261
Not found: 149
Coverage: 89.43 %


In [8]:
qs = list(set(qs))  
qs = sorted(qs)

In [9]:
df_search = pd.DataFrame(results)
df_search.to_csv("vitterhets_search_test.csv", index=False)

In [10]:
end_time = datetime.now()
print("End time:", end_time)

runtime = end_time - start_time
print("Total runtime:", runtime)
seconds = runtime.total_seconds()

print(f"Runtime seconds: {seconds:.1f}")
print(f"Runtime minutes: {seconds/60:.2f}")

End time: 2026-03-05 05:33:15.387672
Total runtime: 0:23:10.068727
Runtime seconds: 1390.1
Runtime minutes: 23.17
